# 03 — Carga da Camada Silver

Este notebook lê a Bronze e produz a Silver com limpeza, padronização e integração.

**Transformações aplicadas**:
- Limpeza e padronização de nomes e tipos
- Tratamento de valores ausentes e registros inválidos
- Deduplicação por chaves de negócio
- Validação de consistência e chaves de relacionamento
- **Integração das bases**: join Indicador + UF + Município + Metas

**Regras de qualidade** embutidas com relatório ao final.

**Próximo passo**: executar `04_carga_camada_gold.py`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Verifica pré-requisitos
tabelas_bronze = [
    "bronze.tc02_uf_raw",
    "bronze.tc02_meta_brasil_raw",
    "bronze.tc02_meta_uf_raw",
    "bronze.tc02_indicador_municipio_raw",
    "bronze.tc02_municipio_raw",
]

for tabela in tabelas_bronze:
    try:
        spark.read.table(tabela).limit(1).count()
    except Exception as err:
        raise ValueError(
            f"Tabela Bronze '{tabela}' não encontrada. Execute 02_carga_camada_bronze.py."
        ) from err

print("Pré-requisitos verificados. Iniciando carga Silver...")

## Leitura das tabelas Bronze

In [0]:
_drop_meta = ["_data_ingestao_bronze", "_fonte", "_sistema_origem", "_data_criacao_origem"]

df_bz_uf        = spark.read.table("bronze.tc02_uf_raw").drop(*_drop_meta)
df_bz_meta_br   = spark.read.table("bronze.tc02_meta_brasil_raw").drop(*_drop_meta)
df_bz_meta_uf   = spark.read.table("bronze.tc02_meta_uf_raw").drop("_data_ingestao_bronze", "_fonte")
df_bz_indicador = spark.read.table("bronze.tc02_indicador_municipio_raw").drop(*_drop_meta)
df_bz_municipio = spark.read.table("bronze.tc02_municipio_raw").drop(*_drop_meta)

## Silver 1: Dimensão UF

In [0]:
df_silver_uf = (
    df_bz_uf
    .withColumn("sigla_uf", F.trim(F.upper(F.col("sigla_uf"))))
    .withColumn("nome_uf",  F.trim(F.col("nome_uf")))
    .withColumn("regiao",   F.trim(F.col("regiao")))
    .dropDuplicates(["sigla_uf"])
    .dropna(subset=["sigla_uf"])
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver UF: {df_silver_uf.count()} registros")
display(df_silver_uf)

## Silver 2: Dimensão Município

In [0]:
df_silver_municipio = (
    df_bz_municipio
    .withColumn("id_municipio",       F.col("id_municipio").cast("int"))
    .withColumn("nome_municipio",     F.trim(F.col("nome_municipio")))
    .withColumn("sigla_uf",           F.trim(F.upper(F.col("sigla_uf"))))
    .withColumn("populacao_estimada", F.col("populacao_estimada").cast("int"))
    .withColumn("capital",            F.col("capital").cast("boolean"))
    .dropDuplicates(["id_municipio"])
    .dropna(subset=["id_municipio", "sigla_uf"])
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Município: {df_silver_municipio.count()} registros")
display(df_silver_municipio.limit(10))

## Silver 3: Meta Nacional

In [0]:
df_silver_meta_brasil = (
    df_bz_meta_br
    .withColumn("ano",                F.col("ano").cast("int"))
    .withColumn("meta_nacional",      F.col("meta_nacional").cast("double"))
    .withColumn("meta_nacional_fracao", F.round(F.col("meta_nacional") / 100.0, 4))
    .dropDuplicates(["ano"])
    .dropna(subset=["ano", "meta_nacional"])
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Meta Brasil: {df_silver_meta_brasil.count()} registros")
display(df_silver_meta_brasil)

## Silver 4: Meta por UF

Valida chave estrangeira: `sigla_uf` deve existir na dimensão UF.

In [0]:
siglas_validas = {row.sigla_uf for row in df_silver_uf.select("sigla_uf").collect()}

df_silver_meta_uf = (
    df_bz_meta_uf
    .withColumn("sigla_uf",       F.trim(F.upper(F.col("sigla_uf"))))
    .withColumn("ano",            F.col("ano").cast("int"))
    .withColumn("meta_uf",        F.col("meta_uf").cast("double"))
    .withColumn("meta_uf_fracao", F.round(F.col("meta_uf") / 100.0, 4))
    .dropDuplicates(["sigla_uf", "ano"])
    .dropna(subset=["sigla_uf", "ano", "meta_uf"])
    .filter(F.col("sigla_uf").isin(siglas_validas))
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Meta UF: {df_silver_meta_uf.count()} registros")
display(df_silver_meta_uf.orderBy("sigla_uf"))

## Silver 5: Fato — Indicador por Município

Aplica todas as regras de qualidade sobre o dataset principal:
- Remove registros com campos obrigatórios nulos
- Filtra indicadores fora do range [0, 100]
- Remove duplicatas por chave de negócio `(id_municipio, ano)`

In [0]:
df_silver_indicador = (
    df_bz_indicador
    .withColumn("id_municipio",                  F.col("id_municipio").cast("int"))
    .withColumn("sigla_uf",                      F.trim(F.upper(F.col("sigla_uf"))))
    .withColumn("ano",                           F.col("ano").cast("int"))
    .withColumn("total_alunos_2o_ano",           F.col("total_alunos_2o_ano").cast("int"))
    .withColumn("alunos_alfabetizados",          F.col("alunos_alfabetizados").cast("int"))
    .withColumn("indicador_crianca_alfabetizada", F.col("indicador_crianca_alfabetizada").cast("double"))
    .withColumn("ponto_corte_saeb",              F.col("ponto_corte_saeb").cast("int"))
    .fillna({"total_alunos_2o_ano": 0, "alunos_alfabetizados": 0})
    .dropna(subset=["id_municipio", "sigla_uf", "ano", "indicador_crianca_alfabetizada"])
    .filter(F.col("indicador_crianca_alfabetizada").between(0, 100))
    .filter(F.col("total_alunos_2o_ano") >= 0)
    .dropDuplicates(["id_municipio", "ano"])
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Indicador Município: {df_silver_indicador.count()} registros (após limpeza)")

## Silver 6: Tabela Integrada

Join do fato principal com todas as dimensões e metas.
Esta tabela é a entrada para a camada Gold.

In [0]:
df_silver_integrado = (
    df_silver_indicador
    .join(
        df_silver_municipio.select("id_municipio", "nome_municipio", "populacao_estimada", "capital"),
        on="id_municipio", how="left"
    )
    .join(
        df_silver_uf.select("sigla_uf", "nome_uf", "regiao"),
        on="sigla_uf", how="left"
    )
    .join(
        df_silver_meta_brasil.select("ano", "meta_nacional"),
        on="ano", how="left"
    )
    .join(
        df_silver_meta_uf.select("sigla_uf", "ano", "meta_uf"),
        on=["sigla_uf", "ano"], how="left"
    )
    .withColumn("regiao",        F.coalesce(F.col("regiao"),        F.lit("NAO_INFORMADA")))
    .withColumn("capital",       F.coalesce(F.col("capital"),       F.lit(False)))
    .withColumn("nome_municipio", F.coalesce(F.col("nome_municipio"), F.col("sigla_uf")))
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Integrado: {df_silver_integrado.count()} registros")
print(f"Colunas: {df_silver_integrado.columns}")
display(df_silver_integrado.orderBy("sigla_uf", "ano").limit(20))

## Relatório de Qualidade de Dados

Verificações aplicadas após a Silver:
- Duplicidade
- Valores ausentes em campos críticos
- Indicador fora do range esperado
- Municípios sem correspondência na dimensão
- Registros sem meta UF cadastrada

In [0]:
total_bronze = spark.read.table("bronze.tc02_indicador_municipio_raw").count()
total_silver = df_silver_indicador.count()

print("=" * 50)
print("RELATÓRIO DE QUALIDADE — CAMADA SILVER")
print("=" * 50)

print(f"\nRegistros Bronze (bruto):  {total_bronze}")
print(f"Registros Silver (limpo):  {total_silver}")
print(f"Descartados:               {total_bronze - total_silver} ({round((total_bronze - total_silver) / total_bronze * 100, 1)}%)")

duplicados_bronze = total_bronze - spark.read.table("bronze.tc02_indicador_municipio_raw").dropDuplicates(["id_municipio", "ano"]).count()
print(f"\nDuplicatas removidas (id_municipio, ano): {duplicados_bronze}")

for col in ["indicador_crianca_alfabetizada", "total_alunos_2o_ano", "sigla_uf"]:
    nulos = df_silver_indicador.filter(F.col(col).isNull()).count()
    pct = round(nulos / total_silver * 100, 2) if total_silver > 0 else 0
    print(f"Nulos em '{col}': {nulos} ({pct}%)")

fora_range = df_silver_indicador.filter(
    (F.col("indicador_crianca_alfabetizada") < 0) | (F.col("indicador_crianca_alfabetizada") > 100)
).count()
print(f"\nIndicador fora do range [0, 100]: {fora_range}")

mun_sem_dim = df_silver_indicador.join(
    df_silver_municipio.select("id_municipio"), on="id_municipio", how="left_anti"
).count()
print(f"Municípios sem dimensão: {mun_sem_dim}")

sem_meta_uf = df_silver_integrado.filter(F.col("meta_uf").isNull()).count()
pct_sem_meta = round(sem_meta_uf / df_silver_integrado.count() * 100, 1)
print(f"Registros sem meta UF: {sem_meta_uf} ({pct_sem_meta}%)")

print("\n" + "=" * 50)
print("QUALIDADE VERIFICADA — Silver pronta para Gold")
print("=" * 50)

## Escrita na Camada Silver

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

tabelas_silver = {
    "silver.tc02_dim_uf":                df_silver_uf,
    "silver.tc02_dim_municipio":         df_silver_municipio,
    "silver.tc02_meta_brasil":           df_silver_meta_brasil,
    "silver.tc02_meta_uf":               df_silver_meta_uf,
    "silver.tc02_indicador_municipio":   df_silver_indicador,
    "silver.tc02_integrado":             df_silver_integrado,
}

for nome, df in tabelas_silver.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )

print("Tabelas Silver criadas com sucesso:")
for nome in tabelas_silver:
    print(f"  - {nome} => {spark.read.table(nome).count()} linhas")

print("\nPróximo passo: executar 04_carga_camada_gold.py")